# Downloading the AGAP airborne gravity survey to use in the notebooks

In [3]:
import bordado as bd
import pandas as pd
import plotly.io as pio
import pooch

import airbornegeo

pio.renderers.default = "notebook"

## load data

In [4]:
path = pooch.retrieve(
    url="https://ramadda.data.bas.ac.uk/repository/entry/get/AGAP_BAS_Grav.XYZ?entryid=synth%3A8e5f910b-11d6-4a9d-bdf7-175c9b98cfb8%3AL0FHQVBfQkFTX0dyYXYuWFla",
    fname="AGAP_BAS_Grav.XYZ",
    path=f"{pooch.os_cache('airbornegeo')}",
    known_hash="391225810f1d15be21b37f506c098960d92af9b3ec0b48bb55dfa20e7b4cf25e",
    progressbar=True,
)
data_df = pd.read_csv(path)
data_df

,/ ------------------------------------------------------------------------------
0,/ XYZ EXPORT [09/03/2014]
1,/ DATABASE [.\Grav\AGAP_BAS_Grav.gdb]
2,/ --------------------------------------------...
3,/
4,/ Line_no Flight_ID Lon ...
...,...
590685,V10320 33 76.910829 ...
590686,V10320 33 76.908151 ...
590687,V10320 33 76.905471 ...
590688,V10320 33 76.902792 ...


In [5]:
skiprow = list(range(11))
skiprow.remove(5)
file = pd.read_csv(path, skiprows=skiprow, sep=" +", engine="python")
for i, col in enumerate(file.columns):
    print(i - 1, col)

-1 /
0 Line_no
1 Flight_ID
2 Lon
3 Lat
4 x
5 y
6 Height_WGS1984
7 Date
8 Time
9 ST
10 CC
11 RB
12 XACC
13 LACC
14 Still
15 Base
16 ST_real
17 Beam_vel
18 rec_grav
19 Abs_grav
20 VaccCor
21 EotvosCor
22 LatCor
23 FaCor
24 HaccCor
25 Free_air
26 FAA_filt
27 FAA_clip
28 Level_cor
29 FAA_level
30 Fa_4600m


In [6]:
data_df = pd.read_csv(
    path,
    skiprows=list(range(11)),
    sep=r"\s+",
    engine="python",
    na_values=["*"],
    names=file.columns[1:],
)

# drop projected coordinates, we will reproject ourselves
data_df = data_df.drop(columns=["x", "y"])

# reproject to EPSG:3031 Polar Stereographic
data_df["easting"], data_df["northing"] = airbornegeo.reproject(
    data_df.Lon,
    data_df.Lat,
    input_crs="EPSG:4326",
    output_crs="EPSG:3031",
)

# subset to region of main suvey
region = (1000e3, 1700e3, -150e3, 700e3)
# data_df = ptk.points_inside_region(data_df, region=region)
inside = bd.inside(coordinates=(data_df.easting, data_df.northing), region=region)
data_df = data_df[inside]

# drop rows with nans in certain columns
data_df = data_df.dropna(subset=["Line_no", "Flight_ID", "Lon", "Lat"], how="any")

# combine flight and line strings together
data_df["line_name"] = data_df.Flight_ID + "_" + data_df.Line_no

# convert line label to integer
data_df["line"] = airbornegeo.unique_line_id(data_df, line_col_name="line_name")

data_df

,Line_no,Flight_ID,Lon,Lat,Height_WGS1984,Date,Time,ST,CC,RB,...,Free_air,FAA_filt,FAA_clip,Level_cor,FAA_level,Fa_4600m,easting,northing,line_name,line
8760,0,18,72.780974,-78.507522,3612.9,2008/12/21,20:34:02.0,12122.71,-1.03,2540.5,...,-1437.8,40.41,40.41,16.57,23.8,23.9,1.196573e+06,370836.746235,18_0,1
8761,0,18,72.782462,-78.507119,3613.0,2008/12/21,20:34:03.0,12122.71,-0.69,2629.3,...,-486.7,40.70,40.70,16.57,24.1,24.1,1.196625e+06,370818.757436,18_0,1
8762,0,18,72.783941,-78.506714,3613.2,2008/12/21,20:34:04.0,12122.71,-1.37,2667.4,...,547.5,40.99,40.99,16.57,24.4,24.4,1.196677e+06,370801.019118,18_0,1
8763,0,18,72.785413,-78.506308,3613.4,2008/12/21,20:34:05.0,12122.71,-1.61,2847.3,...,357.7,41.28,41.28,16.57,24.7,24.7,1.196729e+06,370783.457047,18_0,1
8764,0,18,72.786876,-78.505901,3613.6,2008/12/21,20:34:06.0,12122.71,-1.64,3013.9,...,568.8,41.57,41.57,16.57,25.0,24.9,1.196781e+06,370766.113011,18_0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590675,V10320,33,76.910829,-77.294225,3747.4,2008/12/28,19:25:55.0,12050.72,-1.16,-247.6,...,1103.1,-63.05,-63.05,11.23,-74.3,-72.8,1.349961e+06,313876.801597,33_V10320,137
590676,V10320,33,76.908151,-77.294180,3747.3,2008/12/28,19:25:56.0,12050.72,-1.25,73.5,...,647.9,-62.94,-62.94,11.22,-74.2,-72.7,1.349951e+06,313941.018998,33_V10320,137
590677,V10320,33,76.905471,-77.294137,3747.3,2008/12/28,19:25:57.0,12050.72,-0.21,435.5,...,-1052.9,-62.82,-62.82,11.23,-74.0,-72.6,1.349941e+06,314005.233466,33_V10320,137
590678,V10320,33,76.902792,-77.294094,3747.4,2008/12/28,19:25:58.0,12050.72,0.08,833.1,...,-284.7,-62.70,-62.70,11.23,-73.9,-72.5,1.349931e+06,314069.424118,33_V10320,137


In [7]:
airbornegeo.plotly_points(
    data_df[::10],  # plot every 10th point
    color_col="line",
    hover_cols=["line_name", "Flight_ID"],
    robust=False,
    size=3,
)

In [8]:
# drop a few lines to clean up the survey
data_df = data_df[
    ~data_df.line_name.isin(
        [
            "31_V10290",
            "31_V10310",
            "33_V10300",
            "33_V10320",
            "32_F10250",
            "32_F10270",
            "37_F10290",
            "37_F10291",
            "41_V10250",
            "41_V10270",
            "18_0",
            "19_0",
            "21_0",
            "22_0",
            "23_0",
            "24_0",
            "26_0",
            "27_0",
            "28_0",
            "29_0",
            "30_0",
            "31_0",
            "31_1",
            "32_0",
            "33_0",
            "36_0",
            "37_0",
            "38_0",
            "41_0",
            "42_0",
            "43_0",
            "45_0",
            "47_0",
            "48_0",
            "49_0",
            "54_0",
            "58_0",
        ]
    )
]

In [9]:
# convert supplied line names and flights into integers
data_df["line"] = airbornegeo.unique_line_id(data_df, line_col_name="line_name")

# drop unneeded columns
data_df = data_df.drop(columns=["Line_no", "Flight_ID"])

# drop rows with all NaNs
data_df = data_df.dropna(how="all")

data_df

,Lon,Lat,Height_WGS1984,Date,Time,ST,CC,RB,XACC,LACC,...,Free_air,FAA_filt,FAA_clip,Level_cor,FAA_level,Fa_4600m,easting,northing,line_name,line
93830,77.252450,-80.583923,4156.1,2008/12/17,09:42:48.0,11934.47,2.61,-659.0,-49.0,273.0,...,1186.4,49.38,49.38,7.03,42.4,40.8,1.000024e+06,226237.330771,11_DA500,1
93831,77.252672,-80.583377,4156.0,2008/12/17,09:42:49.0,11934.47,2.72,-368.6,-321.0,230.0,...,342.1,49.45,49.45,7.04,42.4,40.8,1.000083e+06,226246.631269,11_DA500,1
93832,77.252901,-80.582831,4156.1,2008/12/17,09:42:50.0,11888.95,-2.08,703.1,433.0,146.0,...,-1965.9,49.52,49.52,7.04,42.5,40.9,1.000142e+06,226255.809132,11_DA500,1
93833,77.253131,-80.582285,4156.4,2008/12/17,09:42:51.0,11888.95,0.50,625.1,566.0,223.0,...,820.0,49.58,49.58,7.03,42.5,40.9,1.000201e+06,226264.969079,11_DA500,1
93834,77.253358,-80.581740,4156.6,2008/12/17,09:42:52.0,11888.95,-1.73,575.1,108.0,205.0,...,3198.0,49.65,49.65,7.04,42.6,41.0,1.000260e+06,226274.156809,11_DA500,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497680,72.637128,-74.773798,2112.2,2008/12/27,12:42:24.0,12235.41,-7.47,-3702.0,-178.0,-182.0,...,-1466.6,-16.89,-16.89,NaN,NaN,NaN,1.587964e+06,496507.950674,30_E10400,100
497681,72.638964,-74.773861,2111.7,2008/12/27,12:42:25.0,12235.41,-7.15,-3432.2,-286.0,-189.0,...,-1288.3,-16.83,-16.83,NaN,NaN,NaN,1.587973e+06,496454.987752,30_E10400,100
497682,72.640803,-74.773925,2111.2,2008/12/27,12:42:26.0,12235.41,-8.84,-3359.5,-254.0,-207.0,...,1363.5,-16.76,-16.76,NaN,NaN,NaN,1.587983e+06,496401.908627,30_E10400,100
497683,72.642645,-74.773990,2110.6,2008/12/27,12:42:27.0,12235.41,-8.37,-3949.1,-257.0,-163.0,...,1616.2,-16.69,-16.69,NaN,NaN,NaN,1.587992e+06,496348.713306,30_E10400,100


In [10]:
airbornegeo.plotly_points(
    data_df[::10],
    color_col="line",
    hover_cols=["line_name"],
    robust=False,
    size=3,
)

In [11]:
# calculate the unixtime from the date and time columns
data_df["Date"] = pd.to_datetime(data_df["Date"])
data_df["Time"] = pd.to_timedelta(data_df["Time"])
data_df["unixtime"] = data_df["Date"] + data_df["Time"]
data_df = data_df.dropna(subset=["unixtime"])
data_df["unixtime"] = data_df["unixtime"].apply(lambda x: x.timestamp())
data_df = data_df.sort_values(["line", "unixtime"]).reset_index(drop=True)
data_df.head()

,Lon,Lat,Height_WGS1984,Date,Time,ST,CC,RB,XACC,LACC,...,FAA_filt,FAA_clip,Level_cor,FAA_level,Fa_4600m,easting,northing,line_name,line,unixtime
0,77.252450,-80.583923,4156.1,2008-12-17,0 days 09:42:48,11934.47,2.61,-659.0,-49.0,273.0,...,49.38,49.38,7.03,42.4,40.8,1.000024e+06,226237.330771,11_DA500,1,1.229507e+09
1,77.252672,-80.583377,4156.0,2008-12-17,0 days 09:42:49,11934.47,2.72,-368.6,-321.0,230.0,...,49.45,49.45,7.04,42.4,40.8,1.000083e+06,226246.631269,11_DA500,1,1.229507e+09
2,77.252901,-80.582831,4156.1,2008-12-17,0 days 09:42:50,11888.95,-2.08,703.1,433.0,146.0,...,49.52,49.52,7.04,42.5,40.9,1.000142e+06,226255.809132,11_DA500,1,1.229507e+09
3,77.253131,-80.582285,4156.4,2008-12-17,0 days 09:42:51,11888.95,0.50,625.1,566.0,223.0,...,49.58,49.58,7.03,42.5,40.9,1.000201e+06,226264.969079,11_DA500,1,1.229507e+09
4,77.253358,-80.581740,4156.6,2008-12-17,0 days 09:42:52,11888.95,-1.73,575.1,108.0,205.0,...,49.65,49.65,7.04,42.6,41.0,1.000260e+06,226274.156809,11_DA500,1,1.229507e+09


In [12]:
airbornegeo.plotly_points(
    data_df[::10],
    color_col="unixtime",
    hover_cols=["line"],
    robust=False,
    size=3,
)

In [13]:
data_df.to_csv("data/AGAP_gravity_survey.csv", index=False)